# 050_populate_control_tables
Loads every flow definition from `051_populate_control_tables.sql` into the control tables.
The SQL uses `INSERT OVERWRITE`, so re-running replaces the definitions instead of
duplicating them. To add a table: add a script, add a row to the SQL file, re-run this.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

In [ ]:
# Make sure the control tables exist (safe: they are only created if missing)
run_notebook("01_ddl/011_create_data_flow_control_table.ipynb")

In [ ]:
sql_text = (PIPELINE_ROOT / "05_helper_scripts" / "051_populate_control_tables.sql").read_text(encoding="utf-8")
for statement in sql_text.split(";"):
    if "INSERT" in statement:
        spark.sql(statement.replace("{{ctrl}}", CTRL_DATABASE))

spark.sql(f"""
    SELECT layer, COUNT(*) AS flows
    FROM {CTRL_DATABASE}.data_flow_control
    GROUP BY layer
    ORDER BY layer
""").show()
spark.sql(f"SELECT * FROM {CTRL_DATABASE}.source_type_registry ORDER BY layer, source_type").show(truncate=False)